In [1]:
%load_ext autoreload
%autoreload 2

In [10]:
import random
import numpy as np
import torch

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # for multi-GPU
    # Crucial for local determinism
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42) # The answer to life, the universe, and Diffusion

In [2]:
import os
import cv2
import sys
import pandas as pd
from torch.utils.data import Dataset

class FaceDataset(Dataset):
    def __init__(self, csv_path, img_dir, transform=None):
        self.df = pd.read_csv(csv_path)
        self.img_dir = img_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        img_id = int(self.df.iloc[idx]['id'])
        img_path = os.path.join(self.img_dir, f"face-{img_id}.png")
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transform:
            image = self.transform(image=image)["image"]

        return image

In [ ]:
sys.path.append("../")
from torch.utils.data import DataLoader
from src.utils.augmentations import get_train_transform
from src.utils.augmentations import denormalize
from src.models.diffusion import DiffusionModel


device = "cuda"

dataset = FaceDataset(
    csv_path="../data/train.csv",
    img_dir="../data/processed_64",
    transform=get_train_transform(64)
)


In [ ]:
# Train 6 different diffusion models, changing one parameter at a time (max batch size 32)
import torch
import torchvision
import matplotlib.pyplot as plt

# Base config (already trained above)
base_cfg = {"timesteps": 1000, "lr": 5e-5, "batch_size": 32, "epochs": 100}

# Only one parameter changes per config (exclude base config)
configs = [
    {**base_cfg},            # base model
    {**base_cfg, "batch_size": 16},           # lower batch size
    {**base_cfg, "timesteps": 500},            # lower timesteps
    {**base_cfg, "lr": 1e-4},                 # change learning rate
    # {**base_cfg, "epochs": 250},              # change epochs
    {**base_cfg, "beta_end": 0.01},           # less noise injected
]

for i, cfg in enumerate(configs, 1):
    print(f"\nTraining model {i} with timesteps={cfg['timesteps']}, lr={cfg['lr']}, batch_size={cfg['batch_size']}, epochs={cfg['epochs']}" + (f", beta_end={cfg['beta_end']}" if 'beta_end' in cfg else '') + (f", channels={cfg['channels']}" if 'channels' in cfg else ''))
    dataloader = DataLoader(dataset, batch_size=cfg["batch_size"], shuffle=True)
    model_kwargs = {"device": device, "timesteps": cfg["timesteps"], "img_size": 64}
    if "beta_end" in cfg:
        model_kwargs["beta_end"] = cfg["beta_end"]

    model = DiffusionModel(**model_kwargs)
    model.fit(dataloader, epochs=cfg["epochs"], lr=cfg["lr"])
    # Save with parameters in filename
    extra = ""
    if "beta_end" in cfg:
        extra += f"_be{cfg['beta_end']}"

    checkpoint = (
        f"../outputs/Diffusion_images/diffusion_t{cfg['timesteps']}_lr{cfg['lr']}_bs{cfg['batch_size']}_ep{cfg['epochs']}{extra}.pth"
    )
    torch.save({"model": model.model.state_dict(), "ema_model": model.ema_model.state_dict()}, checkpoint)
    print(f"Saved checkpoint: {checkpoint}")

    # Sample 12 images and save grid with same parameters
    samples = model.sample(n=12, use_ema=True)
    samples = (samples.clamp(-1, 1) + 1) / 2
    grid = torchvision.utils.make_grid(samples, nrow=4, padding=2)
    img_path = checkpoint.replace(".pth", "_samples.png")
    torchvision.utils.save_image(grid, img_path)
    print(f"Saved sample grid: {img_path}")


Training model 1 with timesteps=1000, lr=5e-05, batch_size=32, epochs=100


Epoch 100/100: 100%|██████████| 141/141 [00:25<00:00,  5.48it/s, loss=0.0089]


Saved checkpoint: ../models/diffusion_t1000_lr5e-05_bs32_ep100.pth


Sampling: 100%|██████████| 1000/1000 [00:19<00:00, 52.47it/s]


Saved sample grid: ../models/diffusion_t1000_lr5e-05_bs32_ep100_samples.png

Training model 2 with timesteps=1000, lr=5e-05, batch_size=16, epochs=100


...

Saved sample grid: ../models/diffusion_t1000_lr0.0001_bs32_ep100_samples.png

Training model 5 with timesteps=1000, lr=5e-05, batch_size=32, epochs=100, beta_end=0.01


Epoch 100/100: 100%|██████████| 141/141 [00:25<00:00,  5.49it/s, loss=0.0348]


Saved checkpoint: ../models/diffusion_t1000_lr5e-05_bs32_ep100_be0.01.pth


Sampling: 100%|██████████| 1000/1000 [00:19<00:00, 52.48it/s]


Saved sample grid: ../models/diffusion_t1000_lr5e-05_bs32_ep100_be0.01_samples.png


In [13]:
import pandas as pd
import torch
import torchvision
import os
import glob
import gc
from PIL import Image
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.inception import InceptionScore
from IPython.display import display, HTML

# 1. Setup paths and clean start
real_images_dir = "../data/processed_64"
models_dir = "../models"
output_dir = "../outputs/Diffusion_images/"
os.makedirs(output_dir, exist_ok=True)

checkpoints = sorted(glob.glob(os.path.join(models_dir, "*.pth")))
results = []

# 2. PRE-LOAD REAL IMAGES (Do this once to save memory!)
print("Loading reference real images...")
real_imgs_list = []
# We use glob to find only images, skipping system files
real_files = glob.glob(os.path.join(real_images_dir, "*"))[:64] 
for fname in real_files:
    try:
        img = Image.open(fname).convert('RGB').resize((64, 64))
        real_imgs_list.append(torchvision.transforms.ToTensor()(img))
    except: continue

real_imgs_uint8 = (torch.stack(real_imgs_list) * 255).to(torch.uint8).to(device)
del real_imgs_list # Free CPU ram

# 3. EVALUATION LOOP
for ckpt_path in checkpoints:
    print(f"\nEvaluating: {os.path.basename(ckpt_path)}")
    
    # Clear VRAM for the next model
    torch.cuda.empty_cache()
    gc.collect()

    # Set fixed seed for reproducibility (important for fair comparison)
    set_seed(42)

    # Load Model
    # IMPORTANT: Ensure your UNet class definition is in the same notebook
    model = DiffusionModel(device=device, timesteps=1000, img_size=64)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.model.load_state_dict(ckpt["model"])
    model.ema_model.load_state_dict(ckpt["ema_model"])

    with torch.no_grad():
        # Generate 64 samples
        # Fixed Seed note: Since you set the seed earlier, this will 
        # generate the SAME 64 starting points for every model.
        samples = model.sample(n=64, use_ema=True) 
        samples_01 = torch.clamp((samples * 0.5 + 0.5), 0, 1)
        
        # Save Grid
        grid_filename = os.path.basename(ckpt_path).replace(".pth", "_grid.png")
        full_img_path = os.path.join(output_dir, grid_filename)
        grid = torchvision.utils.make_grid(samples_01, nrow=8, padding=2)
        torchvision.utils.save_image(grid, full_img_path)

        # Metrics (uint8)
        fake_imgs_uint8 = (samples_01 * 255).to(torch.uint8)
        
        # Feature 768 is the "sweet spot" for 4GB 3050 cards
        fid_metric = FrechetInceptionDistance(feature=768).to(device)
        fid_metric.update(real_imgs_uint8, real=True)
        fid_metric.update(fake_imgs_uint8, real=False)
        fid_score = fid_metric.compute().item()

        is_metric = InceptionScore().to(device)
        is_metric.update(fake_imgs_uint8)
        is_mean, is_std = is_metric.compute()

        results.append({
            "Model Name": os.path.basename(ckpt_path),
            "FID ↓": round(fid_score, 2),
            "IS ↑": f"{round(is_mean.item(), 2)} ± {round(is_std.item(), 2)}",
            "Sample Grid": full_img_path 
        })

    # Explicit Cleanup
    del model, ckpt, fid_metric, is_metric, samples, samples_01, fake_imgs_uint8

# 4. SHOW RESULTS
df = pd.DataFrame(results)

def show_image(path):
    # This renders the grid in the table. 
    # Use 'relative' path for Jupyter to find the images.
    rel_path = os.path.relpath(path, start=os.getcwd())
    return f'<img src="{rel_path}" width="250" >'

display(HTML(df.to_html(escape=False, formatters={'Sample Grid': show_image})))

Loading reference real images...

Evaluating: diffusion_model_250.pth


Sampling: 100%|██████████| 1000/1000 [01:37<00:00, 10.24it/s]



Evaluating: diffusion_t1000_lr0.0001_bs32_ep100.pth


Sampling: 100%|██████████| 1000/1000 [01:37<00:00, 10.24it/s]



Evaluating: diffusion_t1000_lr5e-05_bs16_ep100.pth


...


Evaluating: diffusion_t1000_lr5e-05_bs32_ep100_be0.01.pth


Sampling: 100%|██████████| 1000/1000 [01:37<00:00, 10.23it/s]



Evaluating: diffusion_t500_lr5e-05_bs32_ep100.pth


Sampling: 100%|██████████| 1000/1000 [01:37<00:00, 10.23it/s]


,Model Name,FID ↓,IS ↑,Sample Grid
0,diffusion_model_250.pth,0.73,1.5 ± 0.37,
1,diffusion_t1000_lr0.0001_bs32_ep100.pth,0.70,1.41 ± 0.22,
2,diffusion_t1000_lr5e-05_bs16_ep100.pth,0.73,1.53 ± 0.35,
3,diffusion_t1000_lr5e-05_bs32_ep100.pth,0.89,1.48 ± 0.29,
4,diffusion_t1000_lr5e-05_bs32_ep100_be0.01.pth,2.08,1.81 ± 0.37,
5,diffusion_t500_lr5e-05_bs32_ep100.pth,4.28,1.84 ± 0.43,


## Final Model Selection & Optimization Strategy


Based on the empirical evidence from the **Ablation Study** (Models 0–5), we have identified the "Golden Configuration" for generating high-fidelity human faces with distinct eyeglasses. While Model 1 ($10^{-4}$ LR) achieved a slightly lower FID at 100 epochs, **Model 2 (Batch Size 16, $5 \times 10^{-5}$ LR)** showed the most promise for long-term "Perfection."

###  Why Batch Size 16 is the "Winning" Choice
Despite the trend toward larger batches in high-compute environments, our results on the **RTX 3050 (4GB)** demonstrate that **Batch Size 16** is superior for this specific task:

1.  **Gradient Noise as Regularizer:** Smaller batches (16 vs 32) introduce helpful stochastic noise during training. This prevents the model from "over-smoothing" facial features, allowing the U-Net to retain the sharp, high-frequency edges required for **eyeglass frames**.
2.  **Superior Convergence:** At the same learning rate ($5 \times 10^{-5}$), the **BS 16** model achieved an **FID of 0.73**, significantly outperforming the **BS 32** model's **0.89**.
3.  **Timestep Precision:** The data confirms that **1000 timesteps** are non-negotiable. Reducing to 500 steps (Model 5) caused the FID to spike to **4.28**, indicating a massive loss in structural integrity.



---

###  Path to Perfection: The 300-Epoch "Marathon"
To move from "identifiable faces" to "photorealistic perfection," we will now scale this configuration to a full **300-epoch** run. 

* **The Logic:** At 100 epochs, the model has learned the *location* of the eyes and glasses. By tripling the iterations, we allow the model to refine the **texture**—moving from blurry dark circles to clear pupils, lens reflections, and symmetrical frames.
* **The Metric Focus:** We will prioritize minimizing **FID (Fréchet Inception Distance)** over **IS (Inception Score)**. Since we are generating a single class (faces), FID is the only reliable mathematical measure of how closely our "specs" match the real dataset's geometry.



---

### ️ The "Perfection" Spec Sheet
* **Architecture:** UNet + Attention + Strided Convolutions
* **Batch Size:** 16
* **Learning Rate:** $5 \times 10^{-5}$ (with Cosine Decay)
* **Timesteps ($T$):** 1000
* **Beta Schedule:** Linear ($1 \times 10^{-4}$ to $0.02$)

---


In [ ]:
dataloader = DataLoader(dataset, 16, shuffle=True)

device = "cuda"

dataset = FaceDataset(
    csv_path="../data/train.csv",
    img_dir="../data/processed_64",
    transform=get_train_transform(64)
)

model_kwargs = {"device": device, "timesteps": 1000, "img_size": 64}
model = DiffusionModel(**model_kwargs)
model.fit(dataloader, epochs=300, lr=5e-5)

checkpoint = "../models/Final/diffusion_final.pth"



In [ ]:
import torchvision

checkpoint = f"../models/Final/diffusion_final.pth"
model = DiffusionModel(device=device, timesteps=1000, img_size=64)
ckpt  = torch.load(checkpoint, map_location=device)
model.model.load_state_dict(ckpt["model"])
model.ema_model.load_state_dict(ckpt["ema_model"])


# --- Sample ---
samples = model.sample(n=16, use_ema=True)
# --- Rescale [-1, 1] → [0, 1] and save grid ---
samples = (samples.clamp(-1, 1) + 1) / 2
grid    = torchvision.utils.make_grid(samples, nrow=4, padding=2)
torchvision.utils.save_image(grid, f"../outputs/final_diffusion_images.png")